# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [8]:
# Write your code below.

%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [9]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [10]:
import os
from glob import glob

# Write your code below.
price_dir = os.getenv("PRICE_DATA")

parquet_files = glob(os.path.join(price_dir, "**", "*.parquet"), recursive=True)
print(f"Found {len(parquet_files)} files")
print(parquet_files[:5])


Found 2725 files
['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet']


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [38]:
# Write your code below.

from dask.dataframe.utils import make_meta

dd_prices = dd.read_parquet(price_dir)

dd_prices = dd_prices.assign(Date=dd.to_datetime(dd_prices['Date']))

keep = ["ticker","Date","Close","Adj Close","High","Low",
        "Close_lag_1","Adj_Close_lag_1","returns","hi_lo_range"]

def add_feats(pdf):
    pdf = pdf.sort_values('Date')
    pdf['Close_lag_1']      = pdf['Close'].shift(1)
    pdf['Adj_Close_lag_1']  = pdf['Adj Close'].shift(1)
    pdf['returns']          = pdf['Close'] / pdf['Close_lag_1'] - 1
    pdf['hi_lo_range']      = pdf['High'] - pdf['Low']
    return pdf[keep]



meta = make_meta({
    'ticker': 'object',
    'Date': 'datetime64[ns]',
    'Close': 'float64',
    'Adj Close': 'float64',
    'High': 'float64',
    'Low': 'float64',
    'Close_lag_1': 'float64',
    'Adj_Close_lag_1': 'float64',
    'returns': 'float64',
    'hi_lo_range': 'float64',
})

dd_feat = (
    dd_prices
    .groupby('ticker', group_keys=False)
    .apply(add_feats, meta=meta)
)


print(dd_feat.head(10, npartitions=-1))


  ticker       Date  Close  Adj Close   High    Low  Close_lag_1  \
0   MOTS 2018-02-14  4.380      4.380  5.280  4.300          NaN   
1   MOTS 2018-02-15  4.300      4.300  4.549  4.300        4.380   
2   MOTS 2018-02-16  4.000      4.000  4.418  4.000        4.300   
3   MOTS 2018-02-20  4.110      4.110  4.550  4.000        4.000   
4   MOTS 2018-02-21  4.250      4.250  4.350  3.750        4.110   
5   MOTS 2018-02-22  4.480      4.480  4.500  4.230        4.250   
6   MOTS 2018-02-23  4.530      4.530  4.580  4.450        4.480   
7   MOTS 2018-02-26  4.756      4.756  5.010  4.589        4.530   
8   MOTS 2018-02-27  5.000      5.000  5.000  4.800        4.756   
9   MOTS 2018-02-28  5.000      5.000  5.130  4.830        5.000   

   Adj_Close_lag_1   returns  hi_lo_range  
0              NaN       NaN        0.980  
1            4.380 -0.018265        0.249  
2            4.300 -0.069767        0.418  
3            4.000  0.027500        0.550  
4            4.110  0.034063   

In [42]:
import pandas as pd

pd_feature = dd_feat.compute()

pd_feature = pd_feature.sort_values(['ticker', 'Date'])

pd_feature ['m_a_r_10d'] = (
    pd_feature.groupby('ticker')['returns']
    .transform(lambda s: s.rolling(10).mean())
)

print(pd_feature.head)

<bound method NDFrame.head of        ticker       Date  Close  Adj Close   High    Low  Close_lag_1  \
137315    ACN 2001-07-19  15.17  11.404394  15.29  15.00          NaN   
137316    ACN 2001-07-20  15.01  11.284108  15.05  14.80        15.17   
137317    ACN 2001-07-23  15.00  11.276587  15.01  14.55        15.01   
137318    ACN 2001-07-24  14.86  11.171341  14.97  14.70        15.00   
137319    ACN 2001-07-25  14.95  11.238999  14.95  14.65        14.86   
...       ...        ...    ...        ...    ...    ...          ...   
88143    ZIXI 2020-03-26   4.51   4.510000   4.53   3.88         4.00   
88144    ZIXI 2020-03-27   4.60   4.600000   4.71   4.10         4.51   
88145    ZIXI 2020-03-30   4.64   4.640000   4.87   4.44         4.60   
88146    ZIXI 2020-03-31   4.31   4.310000   4.69   4.10         4.64   
88147    ZIXI 2020-04-01   3.82   3.820000   4.16   3.80         4.31   

        Adj_Close_lag_1   returns  hi_lo_range  m_a_r_10d  
137315              NaN       NaN

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.